In [ ]:

from phytclust import PhytClust
from phytclust import plot_tree, plot_cluster
from phytclust import pairwise_distances
import phytclust.indices as pc
import io
import os
from Bio import Phylo
import sys
import numpy as np
import random
import matplotlib as mpl
import os
import datetime
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd

# universal_seed = 47  # Example seed
# random.seed(universal_seed)
# np.random.seed(universal_seed)
import csv
from sklearn.metrics.cluster import contingency_matrix
import glob
from sklearn.metrics import (
    adjusted_rand_score,
    v_measure_score,
    f1_score,
    accuracy_score,
    normalized_mutual_info_score,
)
from Bio import Phylo
import string
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder
import io

In [ ]:
tree_path = "/home/ganesank/project/phytclust/simulations_2/data_new/N_16_K_4/tree_40/tree_increase_1.nw"
# tree_path = "/home/ganesank/project/phytclust/example_data/IDC303_tree.newick"
tree = Phylo.read(tree_path, "newick", rooted=True)


# tree = assign_unique_names(tree)

# Phylo.draw(tree)
clust_obj = PhytClust(
    tree, should_plot_scores=True, num_peaks=10
)
# best_peak = clust_obj.best_peaks[0]
print(best_peak)
clust_obj.plot(tree, n=3, height_scale=0.4, marker_size = 70, width_scale = 0.8)
# Phylo.draw(tree)

In [ ]:
import numpy as np
from newick import loads
from prettytable import PrettyTable


# Main method of calculating the number of monophyletic clusterings of a node
def n(node):
    # Terminal nodes must be in 1 cluster, i.e. n_v = [0, 1, 0, 0, ..., 0]
    if len(node.descendants) == 0:
        return [0, 1]
    # Otherwise, we have an internal node, which must be initialized to 0
    # numbers of clusters
    n_v = [1]
    # Product of all the children's number of monophyletic clusterings
    for child in node.descendants:
        n_v = np.convolve(n_v, n(child))
    # Add the possibility that the children are all the same
    # cluster, which has the shape: [0, 1, 0, 0, ..., 0]
    n_v[1] = 1
    return n_v


# Tree is handled by newick module. Here is an example tree
tree = loads(
    "(A:1,(B:1,(C:1,D:1):1):1);"
)
# The tree should only have one root, i.e. len(tree) == 1
assert len(tree) == 1, "Tree has multiple roots (i.e. it's a forest)"

root = tree[0]
k = n(root)
# Remove the 0 entry of the array which corresponds to no clusters at all
k = np.delete(k, 0)

# Create a PrettyTable object
table = PrettyTable()
table.field_names = ["Order (k)", "Number of Monophyletic Clusterings"]

# Add rows to the table
for i, count in enumerate(k, start=1):
    table.add_row([i, count])

# Print the table
print(table)
print(f"Total number of monophyletic clusterings: {k.sum()}")

In [ ]:
import os
from ete3 import Tree
import numpy as np
import collections


def normalized_colless(tree_node) -> float:
    """
    Calculate the normalized Colless index for a phylogenetic tree or subtree.

    Args:
        tree_node (TreeNode): The phylogenetic tree or subtree.

    Returns:
        float: The normalized Colless index.
    """
    # Get the number of terminal nodes (leaves) in the subtree
    n = len(tree_node.get_leaves())

    # Return 0 for singleton clusters or pair clusters, since no imbalance is possible
    if n <= 2:
        return 0

    # Calculate the raw Colless index
    colless_sum = calculate_colless_index(tree_node)

    # Normalize the Colless index
    normalized_colless = (2 * colless_sum) / ((n - 1) * (n - 2))

    return normalized_colless


# Function to calculate raw Colless Index for a tree or subtree
def calculate_colless_index(tree_node):
    if tree_node.is_leaf():
        return 0

    # Get the sizes of the left and right subtrees
    children_sizes = [
        child.get_farthest_leaf(topology_only=True)[1] for child in tree_node.children
    ]

    if len(children_sizes) == 2:
        return abs(children_sizes[0] - children_sizes[1]) + sum(
            calculate_colless_index(child) for child in tree_node.children
        )
    else:
        return sum(calculate_colless_index(child) for child in tree_node.children)


# Parse ground truth file and create cluster lookup
def parse_ground_truth(ground_truth_file):
    cluster_map = {}
    with open(ground_truth_file, "r") as f:
        for line in f:
            node_name, cluster = line.strip().split()
            cluster_map[node_name] = int(cluster)
    return cluster_map


# Calculate stats for each cluster
def calculate_cluster_stats(tree, cluster_map):
    # Group leaves by clusters
    cluster_to_leaves = collections.defaultdict(list)
    for leaf in tree.iter_leaves():
        if leaf.name in cluster_map:
            cluster_to_leaves[cluster_map[leaf.name]].append(leaf)

    cluster_stats = []

    for cluster, leaves in cluster_to_leaves.items():
        subtree = tree.get_common_ancestor(leaves)  # Get the subtree for the cluster
        cluster_size = len(leaves)
        colless_index = normalized_colless(subtree)  # Use normalized Colless index

        cluster_stats.append(
            {
                "cluster": cluster,
                "size": cluster_size,
                "colless_index": colless_index,
            }
        )

    return cluster_stats


# Function to generate the overall statistics file
def generate_stats_file(newick_file, ground_truth_file, output_file):
    # Load the tree
    tree = Tree(newick_file, format=1)

    # Parse ground truth file
    cluster_map = parse_ground_truth(ground_truth_file)

    # Calculate stats for each cluster
    cluster_stats = calculate_cluster_stats(tree, cluster_map)

    # Extract individual statistics for the clusters
    cluster_sizes = [stat["size"] for stat in cluster_stats]
    colless_indices = [stat["colless_index"] for stat in cluster_stats]

    # Calculate mean, median, and balance statistics
    mean_cluster_size = np.mean(cluster_sizes)
    median_cluster_size = np.median(cluster_sizes)
    std_dev_cluster_size = np.std(
        cluster_sizes
    )  # Standard deviation for better differentiation

    average_colless_index = np.mean(colless_indices)

    # Calculate normalized Colless index for the whole tree (supertree balance)
    supertree_colless_index = normalized_colless(tree)

    # Write the stats to the output file
    with open(output_file, "w") as f:
        f.write(f"Mean Cluster Size: {mean_cluster_size}\n")
        f.write(f"Median Cluster Size: {median_cluster_size}\n")
        f.write(f"Standard Deviation of Cluster Size: {std_dev_cluster_size}\n")
        f.write(
            f"Average Normalized Colless Index of Clusters: {average_colless_index}\n"
        )
        f.write(f"Normalized Colless Index of Supertree: {supertree_colless_index}\n")
        f.write("\nCluster Details:\n")
        for stat in cluster_stats:
            f.write(
                f"Cluster {stat['cluster']} - Size: {stat['size']}, Normalized Colless Index: {stat['colless_index']}\n"
            )


# Define the base directory
base_dir = "/home/ganesank/project/phytclust/simulations_2/data/N_100_K_5"

# List all directories in the base directory
all_dirs = os.listdir(base_dir)

# Filter directories to include only those that start with 'tree_'
tree_dirs = [
    d
    for d in all_dirs
    if os.path.isdir(os.path.join(base_dir, d)) and d.startswith("tree_")
]

# Loop over each 'tree_' directory and generate the stats file
for tree_dir in tree_dirs:
    tree_path = os.path.join(base_dir, tree_dir)
    newick_file = os.path.join(tree_path, "tree_increase_0.nw")
    ground_truth_file = os.path.join(tree_path, "ground_truth_labels.txt")
    output_file = os.path.join(tree_path, "stats.txt")

    print(f"Processing directory: {tree_path}")
    generate_stats_file(newick_file, ground_truth_file, output_file)

In [ ]:
import os
from ete3 import Tree
import numpy as np
import collections


def normalized_colless(tree_node) -> float:
    """
    Calculate the normalized Colless index for a phylogenetic tree or subtree.

    Args:
        tree_node (TreeNode): The phylogenetic tree or subtree.

    Returns:
        float: The normalized Colless index.
    """
    # Get the number of terminal nodes (leaves) in the subtree
    n = len(tree_node.get_leaves())

    # Return 0 for singleton clusters or pair clusters, since no imbalance is possible
    if n <= 2:
        return 0

    # Calculate the raw Colless index
    colless_sum = calculate_colless_index(tree_node)

    # Normalize the Colless index
    normalized_colless = (2 * colless_sum) / ((n - 1) * (n - 2))

    return normalized_colless


# Function to calculate raw Colless Index for a tree or subtree
def calculate_colless_index(tree_node):
    if tree_node.is_leaf():
        return 0

    # Get the sizes of the left and right subtrees
    children_sizes = [
        child.get_farthest_leaf(topology_only=True)[1] for child in tree_node.children
    ]

    if len(children_sizes) == 2:
        return abs(children_sizes[0] - children_sizes[1]) + sum(
            calculate_colless_index(child) for child in tree_node.children
        )
    else:
        return sum(calculate_colless_index(child) for child in tree_node.children)


# Parse ground truth file and create cluster lookup
def parse_ground_truth(ground_truth_file):
    cluster_map = {}
    with open(ground_truth_file, "r") as f:
        for line in f:
            node_name, cluster = line.strip().split()
            cluster_map[node_name] = int(cluster)
    return cluster_map


# Calculate stats for each cluster
def calculate_cluster_stats(tree, cluster_map):
    # Group leaves by clusters
    cluster_to_leaves = collections.defaultdict(list)
    for leaf in tree.iter_leaves():
        if leaf.name in cluster_map:
            cluster_to_leaves[cluster_map[leaf.name]].append(leaf)

    cluster_stats = []

    for cluster, leaves in cluster_to_leaves.items():
        subtree = tree.get_common_ancestor(leaves)  # Get the subtree for the cluster
        cluster_size = len(leaves)
        colless_index = normalized_colless(subtree)  # Use normalized Colless index

        cluster_stats.append(
            {
                "cluster": cluster,
                "size": cluster_size,
                "colless_index": colless_index,
            }
        )

    return cluster_stats


# Function to generate the overall statistics file
def generate_stats_file(newick_file, ground_truth_file, output_file):
    # Load the tree
    tree = Tree(newick_file, format=1)

    # Parse ground truth file
    cluster_map = parse_ground_truth(ground_truth_file)

    # Calculate stats for each cluster
    cluster_stats = calculate_cluster_stats(tree, cluster_map)

    # Extract individual statistics for the clusters
    cluster_sizes = [stat["size"] for stat in cluster_stats]
    colless_indices = [stat["colless_index"] for stat in cluster_stats]

    # Calculate mean, median, and balance statistics
    mean_cluster_size = np.mean(cluster_sizes)
    median_cluster_size = np.median(cluster_sizes)
    std_dev_cluster_size = np.std(
        cluster_sizes
    )  # Standard deviation for better differentiation

    average_colless_index = np.mean(colless_indices)

    # Calculate normalized Colless index for the whole tree (supertree balance)
    supertree_colless_index = normalized_colless(tree)

    # Write the stats to the output file
    with open(output_file, "w") as f:
        f.write(f"Mean Cluster Size: {mean_cluster_size}\n")
        f.write(f"Median Cluster Size: {median_cluster_size}\n")
        f.write(f"Standard Deviation of Cluster Size: {std_dev_cluster_size}\n")
        f.write(
            f"Average Normalized Colless Index of Clusters: {average_colless_index}\n"
        )
        f.write(f"Normalized Colless Index of Supertree: {supertree_colless_index}\n")
        f.write("\nCluster Details:\n")
        for stat in cluster_stats:
            f.write(
                f"Cluster {stat['cluster']} - Size: {stat['size']}, Normalized Colless Index: {stat['colless_index']}\n"
            )


# Define the base directory
base_dir = "/home/ganesank/project/phytclust/simulations_2/data_new/N_10_K_3"

# List all directories in the base directory
all_dirs = os.listdir(base_dir)

# Filter directories to include only those that start with 'tree_'
tree_dirs = [
    d
    for d in all_dirs
    if os.path.isdir(os.path.join(base_dir, d)) and d.startswith("tree_")
]

# Loop over each 'tree_' directory and generate the stats file
for tree_dir in tree_dirs:
    tree_path = os.path.join(base_dir, tree_dir)
    newick_file = os.path.join(tree_path, "tree_increase_0.nw")
    ground_truth_file = os.path.join(tree_path, "ground_truth_labels.txt")
    output_file = os.path.join(tree_path, "stats.txt")

    print(f"Processing directory: {tree_path}")
    generate_stats_file(newick_file, ground_truth_file, output_file)

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt


def extract_ari_values(base_dir, is_random_k=False):
    wss_ari_dict = {}
    for tree_folder in os.listdir(base_dir):
        tree_path = os.path.join(base_dir, tree_folder)
        if os.path.isdir(tree_path):
            comparison_files = [
                os.path.join(tree_path, "comparison_results.json"),
                os.path.join(tree_path, "ari_results.json"),
            ]
            for comparison_file in comparison_files:
                if os.path.isfile(comparison_file):
                    with open(comparison_file, "r") as file:
                        data = json.load(file)
                        for entry in data:
                            if "Tree" in entry and "ARI" in entry:
                                tree_name_parts = entry["Tree"].split("_")
                                increase_value = int(tree_name_parts[1])
                                ari_value = entry["ARI"]
                                if increase_value not in wss_ari_dict:
                                    wss_ari_dict[increase_value] = []
                                wss_ari_dict[increase_value].append(
                                    (tree_folder, ari_value)
                                )
                    break  # Stop checking other filenames if one is found
    return wss_ari_dict


def calculate_mean_ci(wss_ari_dict):
    return {
        wss: (
            np.mean([ari for _, ari in ari_list]),
            1.96 * np.std([ari for _, ari in ari_list]) / np.sqrt(len(ari_list)),
        )
        for wss, ari_list in wss_ari_dict.items()
    }


def plot_ari_values(phytclust_ari, random_k_ari, title):
    plt.figure(figsize=(10, 6))

    for data, label in zip(
        [phytclust_ari, random_k_ari], ["PhytClust", "RandomClustering"]
    ):
        sorted_items = sorted(data.items())
        x, y_ci = zip(*sorted_items)
        y, ci = zip(*y_ci)
        plt.errorbar(x, y, yerr=ci, marker="o", label=label, capsize=5)

    plt.xlabel("Increase Factor")
    plt.ylabel("ARI")
    plt.title(title)
    plt.ylim(0, 1.1)
    plt.legend()
    plt.grid(True)
    plt.show()


base_dirs = [
    "/home/ganesank/project/phytclust/simulations_2/results_new/phytclust_single_k",
    # "/home/ganesank/project/phytclust/simulations_2/unbalanced_trees_results/phytclust_all_k",
    "/home/ganesank/project/phytclust/simulations_2/results_new/random_single_k/",
]

for nk_folder in os.listdir(base_dirs[0]):
    phytclust_dir = os.path.join(base_dirs[0], nk_folder)
    random_k_dir = os.path.join(base_dirs[1], nk_folder)

    if os.path.isdir(phytclust_dir) and os.path.isdir(random_k_dir):
        print(phytclust_dir, random_k_dir)
        phytclust_ari_dict = extract_ari_values(phytclust_dir)
        random_k_ari_dict = extract_ari_values(random_k_dir, is_random_k=True)

        phytclust_ari = calculate_mean_ci(phytclust_ari_dict)
        random_k_ari = calculate_mean_ci(random_k_ari_dict)
        print(phytclust_ari_dict)

        # Print increase_0 ARIs with their tree folder names
        if 0 in phytclust_ari_dict:
            print("increase_0 ARIs:")
            for tree_folder, ari_value in phytclust_ari_dict[0]:
                print(f"Tree Folder: {tree_folder}, ARI: {ari_value}")

        plot_ari_values(
            phytclust_ari, random_k_ari, f"Simulation Results for {nk_folder}"
        )

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import collections


def extract_ari_values(base_dir, is_random_k=False):
    wss_ari_dict = {}
    for tree_folder in os.listdir(base_dir):
        tree_path = os.path.join(base_dir, tree_folder)
        if os.path.isdir(tree_path):
            comparison_files = [
                os.path.join(tree_path, "comparison_results.json"),
                os.path.join(tree_path, "ari_results.json"),
            ]
            for comparison_file in comparison_files:
                if os.path.isfile(comparison_file):
                    with open(comparison_file, "r") as file:
                        data = json.load(file)
                        for entry in data:
                            if "Tree" in entry and "ARI" in entry:
                                tree_name_parts = entry["Tree"].split("_")
                                increase_value = int(tree_name_parts[1])
                                ari_value = entry["ARI"]
                                if increase_value not in wss_ari_dict:
                                    wss_ari_dict[increase_value] = []
                                wss_ari_dict[increase_value].append(
                                    (tree_folder, ari_value)
                                )
                    break  # Stop checking other filenames if one is found
    return wss_ari_dict


def calculate_mean_ci(wss_ari_dict):
    return {
        wss: (
            np.mean([ari for _, ari in ari_list]),
            1.96 * np.std([ari for _, ari in ari_list]) / np.sqrt(len(ari_list)),
        )
        for wss, ari_list in wss_ari_dict.items()
    }


def extract_stats(base_dir):
    stats_dict = {}
    for tree_folder in os.listdir(base_dir):
        tree_path = os.path.join(base_dir, tree_folder)
        if os.path.isdir(tree_path):
            stats_file = os.path.join(tree_path, "stats.txt")
            if os.path.isfile(stats_file):
                with open(stats_file, "r") as f:
                    lines = f.readlines()
                    mean_cluster_size = float(lines[0].split(": ")[1])
                    median_cluster_size = float(lines[1].split(": ")[1])
                    std_dev_cluster_size = float(lines[2].split(": ")[1])
                    average_colless_index = float(lines[3].split(": ")[1])
                    supertree_colless_index = float(lines[4].split(": ")[1])
                    stats_dict[tree_folder] = {
                        "mean_cluster_size": mean_cluster_size,
                        "median_cluster_size": median_cluster_size,
                        "std_dev_cluster_size": std_dev_cluster_size,
                        "average_colless_index": average_colless_index,
                        "supertree_colless_index": supertree_colless_index,
                    }
    return stats_dict


def group_by_stat(stats_dict, stat_key):
    grouped = collections.defaultdict(list)
    for tree_folder, stats in stats_dict.items():
        grouped[stats[stat_key]].append(tree_folder)
    return grouped


def plot_ari_by_group(grouped_stats, ari_dict, stat_key):
    plt.figure(figsize=(10, 6))
    for group, tree_folders in grouped_stats.items():
        ari_values = []
        for tree_folder in tree_folders:
            for increase_value, ari_list in ari_dict.items():
                for folder, ari in ari_list:
                    if folder == tree_folder:
                        ari_values.append(ari)
        if ari_values:
            plt.boxplot(ari_values, positions=[group], widths=0.6)
    plt.xlabel(stat_key)
    plt.ylabel("ARI")
    plt.title(f"ARI by {stat_key}")
    plt.grid(True)
    plt.show()


# Define the base directories
phytclust_dir = (
    "/home/ganesank/project/phytclust/simulations_2/results_new/phytclust_single_k/N_100_K_10"
)
random_k_dir = "/home/ganesank/project/phytclust/simulations_2/results_new/random_single_k/N_100_K_10"
phytclust_stats_base_dir = (
    "/home/ganesank/project/phytclust/simulations_2/data_new/N_100_K_10"
)
random_k_stats_base_dir = (
    "/home/ganesank/project/phytclust/simulations_2/data_new/N_100_K_10"
)

# Extract ARI values
phytclust_ari_dict = extract_ari_values(phytclust_dir)
random_k_ari_dict = extract_ari_values(random_k_dir, is_random_k=True)

# Extract stats
phytclust_stats_dict = extract_stats(phytclust_stats_base_dir)
random_k_stats_dict = extract_stats(random_k_stats_base_dir)

In [ ]:
import os
from ete3 import Tree, TreeStyle
import numpy as np
import collections


def normalized_colless(tree_node) -> float:
    """
    Calculate the normalized Colless index for a phylogenetic tree or subtree.

    Args:
        tree_node (TreeNode): The phylogenetic tree or subtree.

    Returns:
        float: The normalized Colless index.
    """
    # Get the number of terminal nodes (leaves) in the subtree
    n = len(tree_node.get_leaves())

    # Return 0 for singleton clusters or pair clusters, since no imbalance is possible
    if n <= 2:
        return 0

    # Calculate the raw Colless index
    colless_sum = calculate_colless_index(tree_node)

    # Normalize the Colless index
    normalized_colless = (2 * colless_sum) / ((n - 1) * (n - 2))

    return normalized_colless


# Function to calculate raw Colless Index for a tree or subtree
def calculate_colless_index(tree_node):
    if tree_node.is_leaf():
        return 0

    # Get the sizes of the left and right subtrees
    children_sizes = [
        child.get_farthest_leaf(topology_only=True)[1] for child in tree_node.children
    ]

    if len(children_sizes) == 2:
        return abs(children_sizes[0] - children_sizes[1]) + sum(
            calculate_colless_index(child) for child in tree_node.children
        )
    else:
        return sum(calculate_colless_index(child) for child in tree_node.children)


# Parse ground truth file and create cluster lookup
def parse_ground_truth(ground_truth_file):
    cluster_map = {}
    with open(ground_truth_file, "r") as f:
        for line in f:
            node_name, cluster = line.strip().split()
            cluster_map[node_name] = int(cluster)
    return cluster_map


# Prune nodes below the MRCA for each cluster
def prune_below_mrca(tree, cluster_map):
    for cluster in set(cluster_map.values()):
        leaves = [
            leaf for leaf in tree.iter_leaves() if cluster_map[leaf.name] == cluster
        ]
        if len(leaves) > 1:
            mrca = tree.get_common_ancestor(leaves)
            for leaf in leaves:
                if leaf.up != mrca:
                    leaf.detach()
    return tree


# Calculate stats for each cluster
def calculate_cluster_stats(tree, cluster_map):
    # Group leaves by clusters
    cluster_to_leaves = collections.defaultdict(list)
    for leaf in tree.iter_leaves():
        if leaf.name in cluster_map:
            cluster_to_leaves[cluster_map[leaf.name]].append(leaf)

    cluster_stats = []

    for cluster, leaves in cluster_to_leaves.items():
        subtree = tree.get_common_ancestor(leaves)  # Get the subtree for the cluster
        cluster_size = len(leaves)
        colless_index = normalized_colless(subtree)  # Use normalized Colless index

        cluster_stats.append(
            {
                "cluster": cluster,
                "size": cluster_size,
                "colless_index": colless_index,
            }
        )

    return cluster_stats


# Function to generate the overall statistics file
def generate_stats_file(newick_file, ground_truth_file, output_file):
    # Load the tree
    tree = Tree(newick_file, format=1)

    # Parse ground truth file
    cluster_map = parse_ground_truth(ground_truth_file)

    # Calculate stats for each cluster
    cluster_stats = calculate_cluster_stats(tree, cluster_map)

    # Extract individual statistics for the clusters
    cluster_sizes = [stat["size"] for stat in cluster_stats]
    colless_indices = [stat["colless_index"] for stat in cluster_stats]

    # Calculate mean, median, and balance statistics
    mean_cluster_size = np.mean(cluster_sizes)
    median_cluster_size = np.median(cluster_sizes)
    std_dev_cluster_size = np.std(
        cluster_sizes
    )  # Standard deviation for better differentiation

    average_colless_index = np.mean(colless_indices)

    # Prune nodes below the MRCA for each cluster
    pruned_tree = prune_below_mrca(tree.copy(), cluster_map)

    # Calculate normalized Colless index for the pruned tree (supertree balance)
    supertree_colless_index = normalized_colless(pruned_tree)

    # Write the stats to the output file
    with open(output_file, "w") as f:
        f.write(f"Mean Cluster Size: {mean_cluster_size}\n")
        f.write(f"Median Cluster Size: {median_cluster_size}\n")
        f.write(f"Standard Deviation of Cluster Size: {std_dev_cluster_size}\n")
        f.write(
            f"Average Normalized Colless Index of Clusters: {average_colless_index}\n"
        )
        f.write(f"Normalized Colless Index of Supertree: {supertree_colless_index}\n")
        f.write("\nCluster Details:\n")
        for stat in cluster_stats:
            f.write(
                f"Cluster {stat['cluster']} - Size: {stat['size']}, Normalized Colless Index: {stat['colless_index']}\n"
            )

    return tree, pruned_tree


# Function to draw and save trees
def draw_and_save_tree(tree, output_file):
    ts = TreeStyle()
    ts.show_leaf_name = True
    tree.render(output_file, tree_style=ts)


# Define the base directory
base_dir = "/home/ganesank/project/phytclust/simulations_2/data_new/N_10_K_3"

# List all directories in the base directory
all_dirs = os.listdir(base_dir)

# Filter directories to include only those that start with 'tree_'
tree_dirs = [
    d
    for d in all_dirs
    if os.path.isdir(os.path.join(base_dir, d)) and d.startswith("tree_")
]

# Loop over each 'tree_' directory and generate the stats file
for i, tree_dir in enumerate(tree_dirs[:3]):  # Process only the first 3 trees
    tree_path = os.path.join(base_dir, tree_dir)
    newick_file = os.path.join(tree_path, "tree_increase_0.nw")
    ground_truth_file = os.path.join(tree_path, "ground_truth_labels.txt")
    output_file = os.path.join(tree_path, "stats.txt")

    print(f"Processing directory: {tree_path}")
    tree, pruned_tree = generate_stats_file(newick_file, ground_truth_file, output_file)

    # Draw and save the unpruned and pruned trees
    draw_and_save_tree(tree, os.path.join(tree_path, "unpruned_tree.png"))
    draw_and_save_tree(pruned_tree, os.path.join(tree_path, "pruned_tree.png"))

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import collections
import random


def extract_ari_values(base_dir, is_random_k=False):
    wss_ari_dict = {}
    for tree_folder in os.listdir(base_dir):
        tree_path = os.path.join(base_dir, tree_folder)
        if os.path.isdir(tree_path):
            comparison_files = [
                os.path.join(tree_path, "vmeasure_results.json")
                # os.path.join(tree_path, "ari_results.json"),
            ]
            for comparison_file in comparison_files:
                if os.path.isfile(comparison_file):
                    with open(comparison_file, "r") as file:
                        data = json.load(file)
                        for entry in data:
                            if "Tree" in entry and "VMeasure" in entry:
                                tree_name_parts = entry["Tree"].split("_")
                                increase_value = int(tree_name_parts[1])
                                ari_value = entry["VMeasure"]
                                if increase_value not in wss_ari_dict:
                                    wss_ari_dict[increase_value] = []
                                wss_ari_dict[increase_value].append(
                                    (tree_folder, ari_value)
                                )
                    break  # Stop checking other filenames if one is found
    return wss_ari_dict


def calculate_mean_ci(wss_ari_dict):
    return {
        wss: (
            np.mean([ari for _, ari in ari_list]),
            1.96 * np.std([ari for _, ari in ari_list]) / np.sqrt(len(ari_list)),
        )
        for wss, ari_list in wss_ari_dict.items()
    }


def extract_stats(base_dir):
    stats_dict = {}
    for tree_folder in os.listdir(base_dir):
        tree_path = os.path.join(base_dir, tree_folder)
        if os.path.isdir(tree_path):
            stats_file = os.path.join(tree_path, "stats.txt")
            if os.path.isfile(stats_file):
                with open(stats_file, "r") as f:
                    lines = f.readlines()
                    mean_cluster_size = float(lines[0].split(": ")[1])
                    median_cluster_size = float(lines[1].split(": ")[1])
                    std_dev_cluster_size = float(lines[2].split(": ")[1])
                    average_colless_index = float(lines[3].split(": ")[1])
                    supertree_colless_index = float(lines[4].split(": ")[1])
                    cluster_sizes = [
                        int(line.split(",")[0].split(": ")[1]) for line in lines[7:]
                    ]
                    num_clusters_size_1 = sum(1 for size in cluster_sizes if size == 1)
                    total_clusters = len(cluster_sizes)
                    percentage_singletons = (num_clusters_size_1 / total_clusters) * 100
                    stats_dict[tree_folder] = {
                        "mean_cluster_size": mean_cluster_size,
                        "median_cluster_size": median_cluster_size,
                        "std_dev_cluster_size": std_dev_cluster_size,
                        "average_colless_index": average_colless_index,
                        "supertree_colless_index": supertree_colless_index,
                        "num_clusters_size_1": num_clusters_size_1,
                        "percentage_singletons": percentage_singletons,
                    }
    return stats_dict


def find_earliest_increase_with_ari_1(ari_dict):
    earliest_increase_dict = {}
    for increase_value, ari_list in ari_dict.items():
        for folder, ari in ari_list:
            if ari == 1.0:
                if (
                    folder not in earliest_increase_dict
                    or increase_value < earliest_increase_dict[folder]
                ):
                    earliest_increase_dict[folder] = increase_value
    return earliest_increase_dict


def plot_ari_vs_increase_by_singletons(ari_dict, stats_dict):
    singleton_groups = collections.defaultdict(list)
    for tree_folder, stats in stats_dict.items():
        percentage_singletons = stats["percentage_singletons"]
        if percentage_singletons <= 20:
            group = "0-20%"
        elif percentage_singletons <= 40:
            group = "20-40%"
        elif percentage_singletons <= 60:
            group = "40-60%"
        elif percentage_singletons <= 80:
            group = "60-80%"
        else:
            group = "80-100%"
        singleton_groups[group].append(tree_folder)

    plt.figure(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0, 1, len(singleton_groups)))
    for color, (group, tree_folders) in zip(colors, singleton_groups.items()):
        increase_to_ari = collections.defaultdict(list)
        for tree_folder in tree_folders:
            for increase_value, ari_list in ari_dict.items():
                for folder, ari in ari_list:
                    if folder == tree_folder:
                        increase_to_ari[increase_value].append(ari)

        increase_values = sorted(increase_to_ari.keys())
        avg_ari_values = [
            np.mean(increase_to_ari[increase]) for increase in increase_values
        ]

        plt.plot(
            increase_values,
            avg_ari_values,
            marker="o",
            label=f"Singletons: {group}",
            color=color,
        )

    plt.xlabel("Increase Value")
    plt.ylabel("VMEasure")
    plt.title("VMeasure vs Increase Value by Percentage of Singleton Clusters")
    plt.legend()
    plt.grid(True)
    plt.show()


# Define the base directories
phytclust_dir = "/home/ganesank/project/phytclust/simulations_2/results/phytclust_results_all_k/N_100_K_80"
# random_k_dir = "/home/ganesank/project/phytclust/simulations_2/results/phylopart_results/N_100_K_80"
phytclust_stats_base_dir = (
    "/home/ganesank/project/phytclust/simulations_2/data/N_100_K_80"
)
# random_k_stats_base_dir = (
#     "/home/ganesank/project/phytclust/simulations_2/data/N_100_K_80"
# )

# Extract ARI values
phytclust_ari_dict = extract_ari_values(phytclust_dir)
# random_k_ari_dict = extract_ari_values(random_k_dir, is_random_k=False)

# Extract stats
phytclust_stats_dict = extract_stats(phytclust_stats_base_dir)

# Plot ARI vs increase value by percentage of singleton clusters for PhytClust
plot_ari_vs_increase_by_singletons(phytclust_ari_dict, phytclust_stats_dict)
# plot_ari_vs_increase_by_singletons(random_k_ari_dict, phytclust_stats_dict)

In [ ]:
def extract_ari_values(directory):
    ari_dict = {}
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith(".json"):
                file_path = os.path.join(root, file)
                with open(file_path, "r") as f:
                    data = json.load(f)
                    for entry in data:
                        if "Tree" in entry and "VMeasure" in entry:
                            tree_name_parts = entry["Tree"].split("_")
                            try:
                                increase_value = int(tree_name_parts[-1])
                            except ValueError:
                                print(f"Skipping invalid tree name: {entry['Tree']}")
                                continue
                            ari_value = entry["VMeasure"]
                            if increase_value not in ari_dict:
                                ari_dict[increase_value] = []
                            ari_dict[increase_value].append(ari_value)
    return ari_dict


# Define the base directories
phytclust_dir = (
    "/home/ganesank/project/phytclust/simulations_2/results/phylopart_results/N_100_K_80"
)
phytclust_stats_base_dir = (
    "/home/ganesank/project/phytclust/simulations_2/data/N_100_K_80"
)

# Extract ARI values
phytclust_ari_dict = extract_ari_values(phytclust_dir)

# Extract stats
phytclust_stats_dict = extract_stats(phytclust_stats_base_dir)

# Plot ARI vs increase value by percentage of singleton clusters for PhytClust
plot_ari_vs_increase_by_singletons(phytclust_ari_dict, phytclust_stats_dict)

In [ ]:
phytclust_ari_dict

In [ ]:
print(
    [
        tree_folder
        for tree_folder, stats in phytclust_stats_dict.items()
        if stats["percentage_singletons"] > 80
    ]
)
high_singleton_trees = [
    tree_folder
    for tree_folder, stats in phytclust_stats_dict.items()
    if stats["percentage_singletons"] > 80
]

plt.figure(figsize=(10, 6))
for tree_folder in high_singleton_trees:
    increase_values = []
    ari_values = []
    for increase_value, ari_list in phytclust_ari_dict.items():
        for folder, ari in ari_list:
            if folder == tree_folder:
                increase_values.append(increase_value)
                ari_values.append(ari)
    if increase_values and ari_values:
        sorted_pairs = sorted(zip(increase_values, ari_values))
        increase_values, ari_values = zip(*sorted_pairs)
        plt.plot(increase_values, ari_values, marker="o", label=tree_folder)

plt.xlabel("Increase Value")
plt.ylabel("ARI")
plt.title("Increase Value vs ARI for Trees with > 80% Singleton Clusters")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import collections
import random


def extract_ari_values(base_dir, is_random_k=False):
    wss_ari_dict = {}
    for tree_folder in os.listdir(base_dir):
        tree_path = os.path.join(base_dir, tree_folder)
        if os.path.isdir(tree_path):
            comparison_files = [
                os.path.join(tree_path, "comparison_results.json"),
                os.path.join(tree_path, "ari_results.json"),
            ]
            for comparison_file in comparison_files:
                if os.path.isfile(comparison_file):
                    with open(comparison_file, "r") as file:
                        data = json.load(file)
                        for entry in data:
                            if "Tree" in entry and "ARI" in entry:
                                tree_name_parts = entry["Tree"].split("_")
                                increase_value = int(tree_name_parts[1])
                                ari_value = entry["ARI"]
                                if increase_value not in wss_ari_dict:
                                    wss_ari_dict[increase_value] = []
                                wss_ari_dict[increase_value].append(
                                    (tree_folder, ari_value)
                                )
                    break  # Stop checking other filenames if one is found
    return wss_ari_dict


def calculate_mean_ci(wss_ari_dict):
    return {
        wss: (
            np.mean([ari for _, ari in ari_list]),
            1.96 * np.std([ari for _, ari in ari_list]) / np.sqrt(len(ari_list)),
        )
        for wss, ari_list in wss_ari_dict.items()
    }


def extract_stats(base_dir):
    stats_dict = {}
    for tree_folder in os.listdir(base_dir):
        tree_path = os.path.join(base_dir, tree_folder)
        if os.path.isdir(tree_path):
            stats_file = os.path.join(tree_path, "stats.txt")
            if os.path.isfile(stats_file):
                with open(stats_file, "r") as f:
                    lines = f.readlines()
                    mean_cluster_size = float(lines[0].split(": ")[1])
                    median_cluster_size = float(lines[1].split(": ")[1])
                    std_dev_cluster_size = float(lines[2].split(": ")[1])
                    average_colless_index = float(lines[3].split(": ")[1])
                    supertree_colless_index = float(lines[4].split(": ")[1])
                    cluster_sizes = [
                        int(line.split(",")[0].split(": ")[1]) for line in lines[7:]
                    ]
                    num_clusters_size_1 = sum(1 for size in cluster_sizes if size == 1)
                    total_clusters = len(cluster_sizes)
                    percentage_singletons = (num_clusters_size_1 / total_clusters) * 100
                    stats_dict[tree_folder] = {
                        "mean_cluster_size": mean_cluster_size,
                        "median_cluster_size": median_cluster_size,
                        "std_dev_cluster_size": std_dev_cluster_size,
                        "average_colless_index": average_colless_index,
                        "supertree_colless_index": supertree_colless_index,
                        "num_clusters_size_1": num_clusters_size_1,
                        "percentage_singletons": percentage_singletons,
                    }
    return stats_dict


def calculate_mean_sd(ari_list):
    mean = np.mean(ari_list)
    sd = np.std(ari_list)
    return mean, sd


def find_earliest_increase_with_ari_1(ari_dict):
    earliest_increase_dict = {}
    for increase_value, ari_list in ari_dict.items():
        for folder, ari in ari_list:
            if ari == 1.0:
                if (
                    folder not in earliest_increase_dict
                    or increase_value < earliest_increase_dict[folder]
                ):
                    earliest_increase_dict[folder] = increase_value
    return earliest_increase_dict


def plot_ari_vs_increase_by_singletons(ari_dict, stats_dict, label_prefix):
    singleton_groups = collections.defaultdict(list)
    for tree_folder, stats in stats_dict.items():
        percentage_singletons = stats["percentage_singletons"]
        if percentage_singletons <= 20:
            group = "0-20%"
        elif percentage_singletons <= 40:
            group = "20-40%"
        elif percentage_singletons <= 60:
            group = "40-60%"
        elif percentage_singletons <= 80:
            group = "60-80%"
        else:
            group = "80-100%"
        singleton_groups[group].append(tree_folder)

    colors = plt.cm.viridis(np.linspace(0, 1, len(singleton_groups)))
    for color, (group, tree_folders) in zip(colors, singleton_groups.items()):
        increase_to_ari = collections.defaultdict(list)
        for tree_folder in tree_folders:
            for increase_value, ari_list in ari_dict.items():
                for folder, ari in ari_list:
                    if folder == tree_folder:
                        increase_to_ari[increase_value].append(ari)

        increase_values = sorted(increase_to_ari.keys())
        avg_ari_values = [
            np.mean(increase_to_ari[increase]) for increase in increase_values
        ]
        mean_sd_values = [
            calculate_mean_sd(increase_to_ari[increase]) for increase in increase_values
        ]
        sd_ari_values = [sd for _, sd in mean_sd_values]

        num_trees = len(tree_folders)
        plt.errorbar(
            increase_values,
            avg_ari_values,
            yerr=sd_ari_values,
            marker="o",
            label=f"{label_prefix} Singletons: {group} (n={num_trees})",
            color=color,
            capsize=5,
        )


# Define the base directories
phytclust_dir = "/home/ganesank/project/phytclust/simulations_2/results_fmi/phytclust_single_k/N_20_K_8"
random_k_dir = (
    "/home/ganesank/project/phytclust/simulations_2/results_new/random_single_k/N_100_K_5"
)
phytclust_stats_base_dir = (
    "/home/ganesank/project/phytclust/simulations_2/data_new/N_20_K_8"
)
random_k_stats_base_dir = (
    "/home/ganesank/project/phytclust/simulations_2/data_new/N_100_K_40"
)

# Extract ARI values
phytclust_ari_dict = extract_ari_values(phytclust_dir)
# random_k_ari_dict = extract_ari_values(random_k_dir, is_random_k=True)

# Extract stats
phytclust_stats_dict = extract_stats(phytclust_stats_base_dir)
# random_k_stats_dict = extract_stats(random_k_stats_base_dir)

# Plot ARI vs increase value by percentage of singleton clusters for PhytClust and Random Clustering
plt.figure(figsize=(10, 6))
plot_ari_vs_increase_by_singletons(
    phytclust_ari_dict, phytclust_stats_dict, "PhytClust"
)
# plot_ari_vs_increase_by_singletons(random_k_ari_dict, random_k_stats_dict, "Random")
plt.xlabel("Increase Value")
plt.ylabel("ARI")
plt.title("ARI vs Increase Value by Percentage of Singleton Clusters")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
random_k_stats_dict

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import collections
import random


def extract_fmi_values(base_dir):
    wss_fmi_dict = {}
    for tree_folder in os.listdir(base_dir):
        tree_path = os.path.join(base_dir, tree_folder)
        if os.path.isdir(tree_path):
            comparison_file = os.path.join(tree_path, "comparison_results.json")
            if os.path.isfile(comparison_file):
                with open(comparison_file, "r") as file:
                    data = json.load(file)
                    for entry in data:
                        if "Tree" in entry and "FMI" in entry:
                            tree_name_parts = entry["Tree"].split("_")
                            increase_value = int(tree_name_parts[1])
                            fmi_value = entry["FMI"]
                            if increase_value not in wss_fmi_dict:
                                wss_fmi_dict[increase_value] = []
                            wss_fmi_dict[increase_value].append(
                                (tree_folder, fmi_value)
                            )
    return wss_fmi_dict


def calculate_mean_ci(wss_fmi_dict):
    return {
        wss: (
            np.mean([fmi for _, fmi in fmi_list]),
            1.96 * np.std([fmi for _, fmi in fmi_list]) / np.sqrt(len(fmi_list)),
        )
        for wss, fmi_list in wss_fmi_dict.items()
    }


def extract_stats(base_dir):
    stats_dict = {}
    for tree_folder in os.listdir(base_dir):
        tree_path = os.path.join(base_dir, tree_folder)
        if os.path.isdir(tree_path):
            stats_file = os.path.join(tree_path, "stats.txt")
            if os.path.isfile(stats_file):
                with open(stats_file, "r") as f:
                    lines = f.readlines()
                    mean_cluster_size = float(lines[0].split(": ")[1])
                    median_cluster_size = float(lines[1].split(": ")[1])
                    std_dev_cluster_size = float(lines[2].split(": ")[1])
                    average_colless_index = float(lines[3].split(": ")[1])
                    supertree_colless_index = float(lines[4].split(": ")[1])
                    cluster_sizes = [
                        int(line.split(",")[0].split(": ")[1]) for line in lines[7:]
                    ]
                    num_clusters_size_1 = sum(1 for size in cluster_sizes if size == 1)
                    total_clusters = len(cluster_sizes)
                    percentage_singletons = (num_clusters_size_1 / total_clusters) * 100
                    stats_dict[tree_folder] = {
                        "mean_cluster_size": mean_cluster_size,
                        "median_cluster_size": median_cluster_size,
                        "std_dev_cluster_size": std_dev_cluster_size,
                        "average_colless_index": average_colless_index,
                        "supertree_colless_index": supertree_colless_index,
                        "num_clusters_size_1": num_clusters_size_1,
                        "percentage_singletons": percentage_singletons,
                    }
    return stats_dict


def plot_fmi_vs_increase_by_singletons(fmi_dict, stats_dict):
    singleton_groups = collections.defaultdict(list)
    for tree_folder, stats in stats_dict.items():
        percentage_singletons = stats["percentage_singletons"]
        if percentage_singletons <= 20:
            group = "0-20%"
        elif percentage_singletons <= 40:
            group = "20-40%"
        elif percentage_singletons <= 60:
            group = "40-60%"
        elif percentage_singletons <= 80:
            group = "60-80%"
        else:
            group = "80-100%"
        singleton_groups[group].append(tree_folder)

    plt.figure(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0, 1, len(singleton_groups)))
    for color, (group, tree_folders) in zip(colors, singleton_groups.items()):
        increase_to_fmi = collections.defaultdict(list)
        for tree_folder in tree_folders:
            for increase_value, fmi_list in fmi_dict.items():
                for folder, fmi in fmi_list:
                    if folder == tree_folder:
                        increase_to_fmi[increase_value].append(fmi)

        increase_values = sorted(increase_to_fmi.keys())
        avg_fmi_values = [
            np.mean(increase_to_fmi[increase]) for increase in increase_values
        ]

        plt.plot(
            increase_values,
            avg_fmi_values,
            marker="o",
            label=f"Singletons: {group}",
            color=color,
        )

    plt.xlabel("Increase Value")
    plt.ylabel("FMI")
    plt.title("FMI vs Increase Value by Percentage of Singleton Clusters")
    plt.legend()
    plt.grid(True)
    plt.show()


# Define the base directories
phytclust_dir = "/home/ganesank/project/phytclust/simulations_2/results_fmi/phytclust_results_all_k/N_20_K_8"
phytclust_stats_base_dir = (
    "/home/ganesank/project/phytclust/simulations_2/data_new/N_20_K_8"
)

# Extract FMI values
phytclust_fmi_dict = extract_fmi_values(phytclust_dir)

# Extract stats
phytclust_stats_dict = extract_stats(phytclust_stats_base_dir)

# Plot FMI vs increase value by percentage of singleton clusters for PhytClust
plot_fmi_vs_increase_by_singletons(phytclust_fmi_dict, phytclust_stats_dict)

In [ ]:
# Import necessary modules from clusim
from clusim.clustering import Clustering
import clusim.sim as sim

# Other imports
from sklearn.metrics import (
    v_measure_score,
    fowlkes_mallows_score,
    adjusted_mutual_info_score,
    normalized_mutual_info_score,
    homogeneity_score,
    completeness_score,
    adjusted_rand_score,
)
from scipy.stats import entropy
from collections import Counter
import numpy as np
from math import log


# Split-Join Distance function
def split_join_distance(true_labels, predicted_labels):
    true_clusters = [
        set(np.where(np.array(true_labels) == label)[0]) for label in set(true_labels)
    ]
    predicted_clusters = [
        set(np.where(np.array(predicted_labels) == label)[0])
        for label in set(predicted_labels)
    ]
    total_points = sum(len(c) for c in true_clusters)
    intersection_matrix = np.zeros((len(true_clusters), len(predicted_clusters)))
    for i, true_cluster in enumerate(true_clusters):
        for j, predicted_cluster in enumerate(predicted_clusters):
            intersection_matrix[i][j] = len(
                true_cluster.intersection(predicted_cluster)
            )
    split = sum(np.max(intersection_matrix, axis=1))
    join = sum(np.max(intersection_matrix, axis=0))
    return 2 * total_points - (split + join)


# Variation of Information (VI) function
def variation_of_information(true_labels, predicted_labels):
    true_clusters = [
        set(np.where(np.array(true_labels) == label)[0]) for label in set(true_labels)
    ]
    predicted_clusters = [
        set(np.where(np.array(predicted_labels) == label)[0])
        for label in set(predicted_labels)
    ]
    n = float(sum([len(x) for x in true_clusters]))  # Total number of elements
    sigma = 0.0
    for x in true_clusters:
        p = len(x) / n  # Probability of cluster x
        for y in predicted_clusters:
            q = len(y) / n  # Probability of cluster y
            r = len(set(x) & set(y)) / n  # Joint probability of x and y overlap
            if r > 0.0:
                sigma += r * (log(r / p, 2) + log(r / q, 2))
    return abs(sigma)


# Overlapping Normalized Mutual Information (ONMI) function using CluSim
def overlapping_nmi(true_labels, predicted_labels):
    # Convert labels to CluSim Clustering objects
    c1 = Clustering(elm2clu_dict={i: [label] for i, label in enumerate(true_labels)})
    c2 = Clustering(
        elm2clu_dict={i: [label] for i, label in enumerate(predicted_labels)}
    )

    # Calculate Overlapping NMI (ONMI)
    return sim.onmi(c1, c2)


# Defining the corrected ground truth and solution labels
ground_truth_labels = ["A"] * 13 + ["N", "O", "P", "Q", "R", "S", "T"]

# Solution 1 labels: {A-D}{E-H}{I-M}{N}{O}{P}{Q}{R}{S}
solution1_labels = (
    ["A"] * 4 + ["E"] * 5 + ["I"] * 4 + ["N", "O", "P", "Q", "R", "S", "T"]
)

# Solution 2 labels: {A-M}{N-S}
solution2_labels = ["A"] * 13 + ["N"] * 7

# Now recalculating various metrics for both solutions
metrics = {
    "V-measure": v_measure_score,
    "FMI": fowlkes_mallows_score,
    "AMI": adjusted_mutual_info_score,
    "NMI": normalized_mutual_info_score,
    "Homogeneity": homogeneity_score,
    "Completeness": completeness_score,
    "ARI": adjusted_rand_score,
    "VI": variation_of_information,
    "SJ": split_join_distance,
    "ONMI": overlapping_nmi,  # Add Overlapping NMI (ONMI)
}

results = {}

# Computing the metrics for both solution 1 and solution 2
for name, metric in metrics.items():
    if name in {"ONMI"}:  # ONMI uses CluSim to handle overlaps
        results[f"{name}_1"] = metric(ground_truth_labels, solution1_labels)
        results[f"{name}_2"] = metric(ground_truth_labels, solution2_labels)
    else:
        results[f"{name}_1"] = metric(ground_truth_labels, solution1_labels)
        results[f"{name}_2"] = metric(ground_truth_labels, solution2_labels)

print(results)

In [ ]:
# Import necessary modules from clusim
from clusim.clustering import Clustering
import clusim.sim as sim

# Other imports
from sklearn.metrics import (
    v_measure_score,
    fowlkes_mallows_score,
    adjusted_mutual_info_score,
    normalized_mutual_info_score,
    homogeneity_score,
    completeness_score,
    adjusted_rand_score,
)
from scipy.stats import entropy
from collections import Counter
import numpy as np
from math import log


# Split-Join Distance function
def split_join_distance(true_labels, predicted_labels):
    true_clusters = [
        set(np.where(np.array(true_labels) == label)[0]) for label in set(true_labels)
    ]
    predicted_clusters = [
        set(np.where(np.array(predicted_labels) == label)[0])
        for label in set(predicted_labels)
    ]
    total_points = sum(len(c) for c in true_clusters)
    intersection_matrix = np.zeros((len(true_clusters), len(predicted_clusters)))
    for i, true_cluster in enumerate(true_clusters):
        for j, predicted_cluster in enumerate(predicted_clusters):
            intersection_matrix[i][j] = len(
                true_cluster.intersection(predicted_cluster)
            )
    split = sum(np.max(intersection_matrix, axis=1))
    join = sum(np.max(intersection_matrix, axis=0))
    return 2 * total_points - (split + join)


# Variation of Information (VI) function
def variation_of_information(true_labels, predicted_labels):
    true_clusters = [
        set(np.where(np.array(true_labels) == label)[0]) for label in set(true_labels)
    ]
    predicted_clusters = [
        set(np.where(np.array(predicted_labels) == label)[0])
        for label in set(predicted_labels)
    ]
    n = float(sum([len(x) for x in true_clusters]))  # Total number of elements
    sigma = 0.0
    for x in true_clusters:
        p = len(x) / n  # Probability of cluster x
        for y in predicted_clusters:
            q = len(y) / n  # Probability of cluster y
            r = len(set(x) & set(y)) / n  # Joint probability of x and y overlap
            if r > 0.0:
                sigma += r * (log(r / p, 2) + log(r / q, 2))
    return abs(sigma)


# Element-centric similarity score function
def element_centric_similarity(true_labels, predicted_labels, alpha=0.9):
    # Convert labels to CluSim Clustering objects
    c1 = Clustering(elm2clu_dict={i: [label] for i, label in enumerate(true_labels)})
    c2 = Clustering(
        elm2clu_dict={i: [label] for i, label in enumerate(predicted_labels)}
    )

    # Calculate element-centric similarity score
    return sim.element_sim(c1, c2, alpha=alpha)


# Jaccard Index function
def jaccard_index(true_labels, predicted_labels):
    true_clusters = [
        set(np.where(np.array(true_labels) == label)[0]) for label in set(true_labels)
    ]
    predicted_clusters = [
        set(np.where(np.array(predicted_labels) == label)[0])
        for label in set(predicted_labels)
    ]
    intersection = sum(
        len(set(x) & set(y)) for x in true_clusters for y in predicted_clusters
    )
    union = sum(len(set(x) | set(y)) for x in true_clusters for y in predicted_clusters)
    return intersection / union


# Purity function
def purity(true_labels, predicted_labels):
    contingency = Counter(zip(true_labels, predicted_labels))
    total = sum(contingency.values())
    max_intersection = sum(
        max(contingency[(k1, k2)] for k2 in set(predicted_labels))
        for k1 in set(true_labels)
    )
    return max_intersection / total


# Entropy function
def clustering_entropy(labels):
    label_counts = Counter(labels)
    total = sum(label_counts.values())
    probs = np.array(list(label_counts.values())) / total
    return entropy(probs)


# Import necessary modules from clusim
from clusim.clustering import Clustering
import clusim.sim as sim

# Other imports
from sklearn.metrics import (
    v_measure_score,
    fowlkes_mallows_score,
    adjusted_mutual_info_score,
    normalized_mutual_info_score,
    homogeneity_score,
    completeness_score,
    adjusted_rand_score,
)
from scipy.stats import entropy
from collections import Counter
import numpy as np
from math import log


# Split-Join Distance function
def split_join_distance(true_labels, predicted_labels):
    true_clusters = [
        set(np.where(np.array(true_labels) == label)[0]) for label in set(true_labels)
    ]
    predicted_clusters = [
        set(np.where(np.array(predicted_labels) == label)[0])
        for label in set(predicted_labels)
    ]
    total_points = sum(len(c) for c in true_clusters)
    intersection_matrix = np.zeros((len(true_clusters), len(predicted_clusters)))
    for i, true_cluster in enumerate(true_clusters):
        for j, predicted_cluster in enumerate(predicted_clusters):
            intersection_matrix[i][j] = len(
                true_cluster.intersection(predicted_cluster)
            )
    split = sum(np.max(intersection_matrix, axis=1))
    join = sum(np.max(intersection_matrix, axis=0))
    return 2 * total_points - (split + join)


# Variation of Information (VI) function
def variation_of_information(true_labels, predicted_labels):
    true_clusters = [
        set(np.where(np.array(true_labels) == label)[0]) for label in set(true_labels)
    ]
    predicted_clusters = [
        set(np.where(np.array(predicted_labels) == label)[0])
        for label in set(predicted_labels)
    ]
    n = float(sum([len(x) for x in true_clusters]))  # Total number of elements
    sigma = 0.0
    for x in true_clusters:
        p = len(x) / n  # Probability of cluster x
        for y in predicted_clusters:
            q = len(y) / n  # Probability of cluster y
            r = len(set(x) & set(y)) / n  # Joint probability of x and y overlap
            if r > 0.0:
                sigma += r * (log(r / p, 2) + log(r / q, 2))
    return abs(sigma)


# Element-centric similarity score function
def element_centric_similarity(true_labels, predicted_labels, alpha=0.9):
    # Convert labels to CluSim Clustering objects
    c1 = Clustering(elm2clu_dict={i: [label] for i, label in enumerate(true_labels)})
    c2 = Clustering(
        elm2clu_dict={i: [label] for i, label in enumerate(predicted_labels)}
    )

    # Calculate element-centric similarity score
    return sim.element_sim(c1, c2, alpha=alpha)


# Jaccard Index function
def jaccard_index(true_labels, predicted_labels):
    true_clusters = [
        set(np.where(np.array(true_labels) == label)[0]) for label in set(true_labels)
    ]
    predicted_clusters = [
        set(np.where(np.array(predicted_labels) == label)[0])
        for label in set(predicted_labels)
    ]
    intersection = sum(
        len(set(x) & set(y)) for x in true_clusters for y in predicted_clusters
    )
    union = sum(len(set(x) | set(y)) for x in true_clusters for y in predicted_clusters)
    return intersection / union


# Purity function
def purity(true_labels, predicted_labels):
    contingency = Counter(zip(true_labels, predicted_labels))
    total = sum(contingency.values())
    max_intersection = sum(
        max(contingency[(k1, k2)] for k2 in set(predicted_labels))
        for k1 in set(true_labels)
    )
    return max_intersection / total


# Entropy function
def clustering_entropy(labels):
    label_counts = Counter(labels)
    total = sum(label_counts.values())
    probs = np.array(list(label_counts.values())) / total
    return entropy(probs)


# Defining the corrected ground truth and solution labels
ground_truth_labels = ["A"] * 4 + ["N", "O", "P", "Q"]

# Solution 1 labels: {A-D}{E-H}{I-M}{N}{O}{P}{Q}{R}{S}
solution1_labels = (
    ["A"] * 1 + ["E"] * 2 + ["I"] * 1 + ["N", "O", "P", "Q"]
)

# Solution 2 labels: {A-M}{N-S}
solution2_labels = ["A"] * 4 + ["N"] * 4

# Now recalculating various metrics for both solutions
metrics = {
    "V-measure": v_measure_score,
    "FMI": fowlkes_mallows_score,
    "AMI": adjusted_mutual_info_score,
    "NMI": normalized_mutual_info_score,
    "Homogeneity": homogeneity_score,
    "Completeness": completeness_score,
    "ARI": adjusted_rand_score,
    "VI": variation_of_information,
    "SJ": split_join_distance,
    "Element-Centric": element_centric_similarity,  # Add element-centric similarity
    "Jaccard": jaccard_index,
    "Purity": purity,
    "Entropy": clustering_entropy,
}

results = {}

# Computing the metrics for both solution 1 and solution 2
for name, metric in metrics.items():
    if name == "Element-Centric":  # Element-centric similarity takes an alpha parameter
        results[f"{name}_1"] = metric(ground_truth_labels, solution1_labels, alpha=0.9)
        results[f"{name}_2"] = metric(ground_truth_labels, solution2_labels, alpha=0.9)
    elif name == "Entropy":  # Entropy is calculated separately for each set of labels
        results[f"{name}_1"] = metric(solution1_labels)
        results[f"{name}_2"] = metric(solution2_labels)
    else:
        results[f"{name}_1"] = metric(ground_truth_labels, solution1_labels)
        results[f"{name}_2"] = metric(ground_truth_labels, solution2_labels)

results

In [ ]:
# Defining the corrected ground truth and solution labels
ground_truth_labels = ["A"] * 4 + ["B", "C", "D"]
# Solution 1 labels: {A-D}{E-H}{I-M}{N}{O}{P}{Q}{R}{S}
solution1_labels = ["A"] * 4 + ["E"] * 3

# Solution 2 labels: {A-M}{N-S}
solution2_labels = ["A"] * 2 + ["B"] * 2 + ["C", "D", "E"]

# Now recalculating various metrics for both solutions
metrics = {
    "V-measure": v_measure_score,
    "FMI": fowlkes_mallows_score,
    "AMI": adjusted_mutual_info_score,
    "NMI": normalized_mutual_info_score,
    "Homogeneity": homogeneity_score,
    "Completeness": completeness_score,
    "ARI": adjusted_rand_score,
    "VI": variation_of_information,
    "SJ": split_join_distance,
    "Element-Centric": element_centric_similarity,  # Add element-centric similarity
    "Jaccard": jaccard_index,
    "Purity": purity,
    "Entropy": clustering_entropy,
}

results = {}

# Computing the metrics for both solution 1 and solution 2
for name, metric in metrics.items():
    if name == "Element-Centric":  # Element-centric similarity takes an alpha parameter
        results[f"{name}_1"] = metric(ground_truth_labels, solution1_labels, alpha=0.9)
        results[f"{name}_2"] = metric(ground_truth_labels, solution2_labels, alpha=0.9)
    elif name == "Entropy":  # Entropy is calculated separately for each set of labels
        results[f"{name}_1"] = metric(solution1_labels)
        results[f"{name}_2"] = metric(solution2_labels)
    else:
        results[f"{name}_1"] = metric(ground_truth_labels, solution1_labels)
        results[f"{name}_2"] = metric(ground_truth_labels, solution2_labels)

results